# SPOTPY calibration steps

**Code under development**
Note: sampler needs set-up and debugging

## Setup

1. Load raven files to Data/Calibration/{model name}


In [1]:
import os
import sys
import pandas as pd
import spotpy as sp

# Ensure the project directory is in the Python path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)
import src.py.raven_py as rv
from src.py.proj_conf import *
from src.py.setup_spotpy import *

import matplotlib.pyplot as plt
import numpy as np

In [2]:
# DATA_CAL = "../Data/Calibration/cypress"
# Check for Raven.exe in the DATA_CAL folder
raven_exe_path = os.path.join(DATA_CAL, "Raven.exe")
raven_name = "Cypress_New"
# raven_name = "CypressRiverT_ERA5"

In [3]:
# Set calibration directory
DATA_CAL_NEW = os.path.join(prj_path, r"Data\Calibration", raven_name).replace('\\','/')

# Specify the calibration results file name and location (no .csv extension)
cal_data_path = DATA_CAL + "../Results/rope_rdrs_offset_20251009_14"

In [4]:
# Specify necessary functions

def generate_param_tuples(spotpy_results, n_params = 7):
    # Generate tuples for parameter pairwise plots
    if isinstance(spotpy_results, pd.DataFrame):
        parameters = spotpy_results.columns[1:n_params + 1]
    else:
        parameters = spotpy_results.dtype.names[1:n_params + 1]
    parameter_tuples = [(parameters[i], parameters[j]) for i in range(len(parameters)) for j in range(i + 1, len(parameters))]
    return parameter_tuples

def plot_param_dist(_data, _param, subset_name):
    # Plot distributions of calibration parameters
    plt.hist(_data[_param], bins=10, alpha=0.7, label=f'count {_param}')
    plt.legend()
    plt.title(f'Distribution of {_param} in top 500 of the {subset_name} distribution')
    median_col = np.round(np.median(_data[_param]), 4)
    mean_col = np.round(np.mean(_data[_param]), 4)
    plt.xlabel(f'median: {median_col}, mean: {mean_col}')
    plt.show()

def sensitivity_plot(_data, x_str, y_str, objective_str, filename = None):
    # Plot objective function values (color) against 2 calibration parameters
    
    # Extract the first 3 columns from _data
    x = _data[x_str]
    y = _data[y_str]
    objective = _data[objective_str]

    # Normalize objective for color mapping
    norm_objective = (objective - objective.min()) / (objective.max() - objective.min())

    # Create a color map: blue for low objective, red for high objective
    colors = plt.cm.coolwarm(norm_objective)

    plt.figure(figsize=(8,6))
    scatter = plt.scatter(x, y, c=colors, s=40, edgecolor='k')
    plt.xlabel(x_str)
    plt.ylabel(y_str)
    plt.title(f'FAST Results: {x_str} vs {y_str} colored by {objective_str}')

    # Add colorbar to show objective scale
    sm = plt.cm.ScalarMappable(cmap=plt.cm.coolwarm, norm=plt.Normalize(vmin=objective.min(), vmax=objective.max()))
    sm.set_array([])
    # cbar = plt.colorbar(sm)
    # cbar.set_label('objective')

    if filename:
        plt.savefig(filename)
        print(f"Plot saved as {filename}")

    plt.show()

## Raven test run

Test the Raven script in the main calibration path

In [ ]:
## Checking end of nc timeseries for RDRS

# from netCDF4 import Dataset

# nc_file = Dataset(DATA_CAL+"input/rdrs_runoff_wlat" + ".nc", "r")

# nc_file.variables["time"][:].min() / 8766

# nc_file.variables["time"][:].max() / 8766


In [5]:
# raven_name = input("Enter the Raven model name (if different):")
if os.path.isfile(raven_exe_path):
    test_confirm = input(f"Raven.exe found at {raven_exe_path}. Do you want to run it? (yes or y to confirm): ")
    if (test_confirm.lower() != 'yes' and test_confirm.lower() != 'y'):
        print("Exiting without running Raven.exe.")
    else:
        print(f"Testing Raven.exe at {raven_exe_path}...")
        # Set working directory to DATA_CAL and run Raven.exe
        rv.run_raven(raven_name, DATA_CAL)
else:
    print(f"Raven.exe not found in location: {DATA_CAL}")

Testing Raven.exe at ../Data/Calibration/Cypress_New/Raven.exe...
Running: Raven.exe Cypress_New -o ./output/
STDOUT:
                        RAVEN                               
 a robust semi-distributed hydrological modelling framework 
    Copyright 2008-2024, the Raven Development Team 
                    Version 3.8.0 w/ netCDF
                BuildDate Jan 10 2024
Generating Master Parameter List...
Autocalculating Model Parameters...
...done Autocalculating.
Checking for Required Model Parameters...
...Done Checking
...model input successfully parsed

Initializing Model...
  Generating Gauge Interpolation Weights...
  Calculating basin & watershed areas...
  Calculating routing network topology...
  Initializing Basins, calculating watershed area, setting initial flow conditions...
  Calculating initial system water storage...
==MODEL SUMMARY=======================================
       Model Run: Cypress_New
      Start time: 1990-01-01
        End time: 2010-12-31 (duration

## Calibration set-up and test run

Test Calibration set-up (without mpi parallelization)

In [6]:

calibrator = spot_setup_rvn(raven_name, DATA_CAL_NEW, obj_fn= "nse_pbias", split_areas = "none", \
                            spring_eval= True, low_seep=False, routing=True, dep_scaling=True, runoff_scaling=True, channel_scaling = True,
                            mpi_cores=0)


Outlet subid from WSC is:  310019543
Aspect ratio path is:  c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New\input\aspect_ratios.csv
HRU areas path is:  c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New\input\Cypress_New_HRU_areas.csv
TP TC path is:  c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New\input\Cypress_New_tp_tc.csv
Deps shapefile is:  c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New\input\CLRH_basins_depression_depths.shp
Directory set-up complete. Core paths for SpotPy calibration: ['c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/']


In [7]:
# Use the Pareto Archived Dynamically Dimensioned Search (PADDS) algorithm for calibration
# sampler=sp.algorithms.padds(spot_setup=calibrator, # Core model setup object
#                             dbname = "padds_raven", # Save results of cal run to this database
#                             dbformat='csv')

 
sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
                           parallel='seq',
                           dbname='test',
                           dbformat='csv')

# sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
#                            parallel='seq',
#                            dbname='mc_raven',
#                            dbformat='csv')



Generated parameters for SpotPy calibration: [( 0.57395367, 'dep_fact',  0.09035,  0.5   , 1.000e-01,   1. , False)
 ( 0.25834455, 'seep_k',  0.04972,  0.25  , 5.001e-03,   0.5, False)
 ( 1.05146054, 'basin_fact',  0.02984,  1.1   , 9.000e-01,   1.2, False)
 ( 2.9038993 , 'tc_tp_fact',  0.1606 ,  2.198 , 1.400e+00,   3. , False)
 ( 1.15464155, 't_peak_fact',  0.08007,  0.8028, 4.000e-01,   1.2, False)
 ( 0.73709674, 'qref_scale',  0.1475 ,  1.249 , 5.000e-01,   2. , False)
 ( 1.10488675, 'bedslope_mult',  0.42   ,  1.5   , 8.000e-01,   5. , False)
 (37.16852598, 'runoff_offset', 10.02   , 60.    , 6.653e-04, 100. , False)]


In [8]:
rep = 3
sampler.sample(rep)

## REMINDER - try to reduce the evaluation period to 2003 onward to improve performance

# setup = calibrator  # your spot_setup object

# params = [p.optguess for p in setup.parameters]
# print("params:", params)

# sim = setup.simulation(params)
# evals = setup.evaluation()
# obj = setup.objectivefunction(sim, evals)
# print("Objective output:", obj)

Initializing the  Monte Carlo (MC) sampler  with  3  repetitions
Starting the MC algorithm with 3 repetitions...
Generated parameters for SpotPy calibration: [( 0.48281935, 'dep_fact',  0.09035,  0.5   , 1.000e-01,   1. , False)
 ( 0.04099408, 'seep_k',  0.04972,  0.25  , 5.001e-03,   0.5, False)
 ( 1.14742386, 'basin_fact',  0.02984,  1.1   , 9.000e-01,   1.2, False)
 ( 2.60778651, 'tc_tp_fact',  0.1606 ,  2.198 , 1.400e+00,   3. , False)
 ( 1.06881218, 't_peak_fact',  0.08007,  0.8028, 4.000e-01,   1.2, False)
 ( 1.55698894, 'qref_scale',  0.1475 ,  1.249 , 5.000e-01,   2. , False)
 ( 4.02255998, 'bedslope_mult',  0.42   ,  1.5   , 8.000e-01,   5. , False)
 (18.63227515, 'runoff_offset', 10.02   , 60.    , 6.653e-04, 100. , False)]
Using PhyDAP template file: c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/input/cypress_rdrs_runoff_tpl.nc
Closed file c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/input/cypress_rdrs_runoff_adj

C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:363: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/output/Cypress_New_Hydrographs.csv
looking for hydrograph file in c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/


C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:363: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/output/Cypress_New_Hydrographs.csv
looking for hydrograph file in c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/


...

Loaded hydrograph data from c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/output/Cypress_New_Hydrographs.csv
looking for hydrograph file in c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/


C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:363: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/output/Cypress_New_Hydrographs.csv
looking for hydrograph file in c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/
Loaded hydrograph data from c:/Users/tsaleh/Code_Workspace/GIS-Projects/PWA/Data/Calibration/Cypress_New/output/Cypress_New_Hydrographs.csv
Simulation: 0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
58075    0.0
58076    0.0
58077    0.0
58078    0.0
58079    0.0
Name: sub310019543 [m3/s], Length: 58080, dtype: float64


C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:363: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Evaluation: 0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
58075   NaN
58076   NaN
58077   NaN
58078   NaN
58079   NaN
Name: sub310019543 (observed) [m3/s], Length: 58080, dtype: float64
Simulation type: <class 'pandas.core.series.Series'>
Evaluation type: <class 'pandas.core.series.Series'>
Nash-Sutcliffe Efficiency (NSE): 0   -0.019615
dtype: float64
Percent Bias (PBIAS): 0   -0.145762
dtype: float64 %
Weighted Objective (2*(NSE-1) - sqrt(PBIAS^2)): 0   -2.184993
dtype: float64
parameters: dep_fact         0.216901
seep_k           0.157802
basin_fact       0.996396
tc_tp_fact       2.353548
t_peak_fact      0.820512
qref_scale       0.626325
bedslope_mult    1.503032
runoff_offset    7.853319
dtype: float64
3 of 3, min objf=-2.22888, max objf=-1.8451, time remaining: 23:58:41

*** Final SPOTPY summary ***
Total Duration: 317.74 seconds
Total Repetitions: 3
Minimal objective value: -2.22888
Corresponding parameter setting:
dep_fact: 0.575729
seep_k: 0.476874


## FAST Analysis

Run fast method in mpiexec

With 5 parameters, the number of reps required for FAST is 2245

In [ ]:
# Load FAST results and plot sensitivity

fast_results = sp.analyser.load_csv_results(cal_data_path)
sp.analyser.plot_fast_sensitivity(fast_results, number_of_sensitiv_pars=3)

In [ ]:
fast_results = fast_results[['like1', 'pardep_fact', 'parseep_k', 'part_peak_fact']]

parameter_tuples = generate_param_tuples(fast_results, n_params = 7)

for param1, param2 in parameter_tuples:
    sensitivity_plot(fast_results, param1, param2, 'like1') # to save, add <, filename=f"fast_sensitivity_{param1}_{param2}.png">

## Calibration Analysis

### Define calibration subset

In [ ]:
calibration_results = sp.analyser.load_csv_results(cal_data_path)

In [ ]:
# Only relevant for a converged calibration
# Compare the prior (first 2500) and posterior (last 2500) distributions of the Spotpy results

calibration_prior = calibration_results[:2500]
calibration_post = calibration_results[-2500:]

plot_param_dist(calibration_prior, 'like1', 'prior')
plot_param_dist(calibration_post, 'like1', 'post')

In [ ]:
# Some options for selecting the best results for analysis

# calibration_post_best = pd.DataFrame(calibration_results[7500:]).sort_values(by='like1', ascending=False).head(500)

calibration_post_best = pd.DataFrame(calibration_post).sort_values(by='like1', ascending=False).head(100)

# calibration_post_best = pd.DataFrame(calibration_results).sort_values(by='like1', ascending=False).head(200)

### Evaluate prior or post subsets

In [ ]:
# Rename the objective function column for clarity
obj_fn_str = "nse-1-norm(pbias)"
calibration_post_best.rename(columns = {"like1" : obj_fn_str}, inplace=True)

def plot_calibration_distributions(calibration_dataset: pd.DataFrame):
    """
    Plots histograms for each column in the calibration dataset,
    showing the distribution, median, and mean for each parameter.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    for col in calibration_dataset.columns:
        plt.hist(calibration_dataset[col], bins=10, alpha=0.7, label=f'count {col}')
        plt.legend()
        plt.title(f'Distribution of {col} in top {len(calibration_dataset)} of the final ROPE simulations')
        median_col = np.round(calibration_dataset[col].median(), 4)
        mean_col = np.round(calibration_dataset[col].mean(), 4)
        plt.xlabel(f'median: {median_col}, mean: {mean_col}')
        # plt.savefig(f"post_{col}.jpg")
        plt.show()

plot_calibration_distributions(pd.DataFrame(calibration_post))

In [ ]:
parameter_tuples = generate_param_tuples(calibration_post_best, n_params = 8)

# View correlations between parameters in the best results
for param1, param2 in parameter_tuples:
        pearson_corr = calibration_post_best[[param1, param2]].corr().iloc[0, 1]
        if np.abs(pearson_corr) > 0.3:
            print(f'Parameter 1: {param1}, Parameter 2: {param2}, corr:{pearson_corr}')

In [ ]:
# Generate heat bi-plots of best results
for param1, param2 in parameter_tuples:
    sensitivity_plot(calibration_post_best, param1, param2, 'nse-1-norm(pbias)') # to save, add <, filename=f"fast_sensitivity_{param1}_{param2}.png">